# Unit 3 Assignment: Building a Production Advanced RAG System
Topic: Advanced RAG — Retrieval Enhancement, Re-Ranking, and Query Expansion

Estimated Time: 60–90 Minutes

Tools: Python, HuggingFace, Groq API, Google Gemini API, rank-bm25, sentence-transformers

In [1]:
!pip install sentence-transformers rank-bm25 google-generativeai groq
!pip install --upgrade google-generativeai

This cell installs all required libraries for building the Advanced RAG pipeline, including:

- sentence-transformers for embeddings
- rank-bm25 for keyword-based retrieval
- google-generativeai for query expansion and answer generation
- Numerical operations using NumPy
- BM25 for sparse retrieval
- Sentence Transformers for dense retrieval
- Cross-Encoder for re-ranking
- Gemini API for query expansion and generation

In [2]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from typing import List, Dict

This cell securely loads the Gemini API key using getpass and initializes the generative model.
This model is later used for:

- Query expansion
- Final answer generation

In [11]:
import google.generativeai as genai
import getpass

api_key = getpass.getpass("Enter your API key: ")
genai.configure(api_key=api_key)

gemini_model = genai.GenerativeModel("gemini-2.5-flash")

Enter your API key: ··········


This cell defines the document corpus used for retrieval.
The corpus contains multiple AI/ML-related concepts, ensuring:

- Coverage of different subtopics
- Inclusion of both general and technical terms

In [12]:
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "Gradient descent is an optimization algorithm used to minimize loss functions.",
    "Adam optimizer combines momentum and adaptive learning rates.",
    "Backpropagation computes gradients for neural network training.",
    "Attention allows models to focus on important tokens in a sequence.",
    "BERT is a bidirectional transformer encoder model.",
    "Dropout is a regularization technique to prevent overfitting.",
    "Convolutional Neural Networks are used for image processing tasks.",
    "Tokenization converts raw text into numerical representations.",
    "The vanishing gradient problem affects deep neural network training.",
    "ReLU is a non-linear activation function widely used in deep learning."
]

This cell implements the HybridRetriever class, which combines:

- BM25 (keyword-based retrieval)
- SBERT (semantic retrieval)

The results are fused using Reciprocal Rank Fusion (RRF) to improve retrieval quality by leveraging both lexical and semantic signals.

In [13]:
class HybridRetriever:
    def __init__(self, corpus: List[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # BM25
        self.tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # SBERT
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_embeddings = self.sbert.encode(corpus, convert_to_tensor=True)

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        query_tokens = query.lower().split()

        # BM25 scores
        bm25_scores = self.bm25.get_scores(query_tokens)
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        # SBERT scores
        query_embedding = self.sbert.encode(query, convert_to_tensor=True)
        sbert_scores = (self.doc_embeddings @ query_embedding).cpu().numpy()
        sbert_ranks = np.argsort(sbert_scores)[::-1]

        # RRF Fusion
        scores = {}
        for i, doc_id in enumerate(bm25_ranks):
            scores.setdefault(doc_id, 0)
            scores[doc_id] += 1 / (self.k + i + 1)

        for i, doc_id in enumerate(sbert_ranks):
            scores.setdefault(doc_id, 0)
            scores[doc_id] += 1 / (self.k + i + 1)

        # Sort by RRF
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for rank, (doc_id, rrf_score) in enumerate(ranked[:top_k]):
            results.append({
                "doc_id": doc_id,
                "rrf_score": rrf_score,
                "bm25_rank": int(np.where(bm25_ranks == doc_id)[0][0] + 1),
                "sbert_rank": int(np.where(sbert_ranks == doc_id)[0][0] + 1),
                "text": self.corpus[doc_id]
            })

        return results

This cell defines a re-ranking function using a Cross-Encoder model.
Unlike bi-encoders, the Cross-Encoder evaluates:

- Query and document together

This improves ranking accuracy by selecting the most relevant documents from retrieved candidates.

In [14]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: List[Dict], top_k: int = 3):
    pairs = [(query, doc["text"]) for doc in candidates]
    scores = cross_encoder.predict(pairs)

    for i, doc in enumerate(candidates):
        doc["cross_score"] = float(scores[i])

    reranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return reranked[:top_k]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


This cell implements query expansion using the Multi-Query approach.
The Gemini model generates multiple paraphrases of the user query to:

- Reduce vocabulary mismatch
- Improve retrieval coverage

In [15]:
def expand_query(query: str) -> List[str]:
    prompt = f"""
    Generate 3 different paraphrases of the following query:
    Query: {query}
    """

    response = gemini_model.generate_content(prompt)

    queries = response.text.strip().split("\n")
    queries = [q.strip("- ").strip() for q in queries if q.strip()]

    return list(set(queries))

This cell defines the baseline Naïve RAG system using only:

- Dense retrieval (SBERT cosine similarity)

It does not include:

- Query expansion
- Hybrid retrieval
- Re-ranking

This serves as a comparison baseline.

In [16]:
def naive_rag(query: str, retriever: HybridRetriever):
    query_embedding = retriever.sbert.encode(query, convert_to_tensor=True)
    scores = (retriever.doc_embeddings @ query_embedding).cpu().numpy()

    top_doc_id = np.argmax(scores)
    return retriever.corpus[top_doc_id]

This cell integrates all components into a complete pipeline:

1. Query Expansion
2. Hybrid Retrieval (BM25 + SBERT + RRF)
3. Re-Ranking (Cross-Encoder)
4. Answer Generation (Gemini)

This pipeline produces more accurate and context-aware responses.

In [17]:
def advanced_rag(user_query: str, retriever: HybridRetriever):

    # Step 1: Query Expansion
    expanded_queries = expand_query(user_query)

    all_candidates = []
    seen_texts = set()

    # Step 2: Retrieval for each query
    for q in expanded_queries:
        results = retriever.retrieve(q, top_k=5)

        for r in results:
            if r["text"] not in seen_texts:
                seen_texts.add(r["text"])
                all_candidates.append(r)

    # Step 3: Re-ranking
    top_docs = rerank(user_query, all_candidates, top_k=3)

    # Step 4: Final Answer Generation
    context = "\n".join([doc["text"] for doc in top_docs])

    prompt = f"""
    Answer the question using the context below:

    Context:
    {context}

    Question:
    {user_query}
    """

    response = gemini_model.generate_content(prompt)

    return response.text, top_docs

This cell evaluates both Naïve RAG and Advanced RAG on multiple queries.
It compares:

- Top retrieved document
- Whether results differ

This helps demonstrate the effectiveness of the Advanced RAG pipeline.

In [18]:
retriever = HybridRetriever(corpus)

queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is attention in deep learning?"
]

results = []

for q in queries:
    naive = naive_rag(q, retriever)
    advanced_answer, advanced_docs = advanced_rag(q, retriever)

    results.append({
        "query": q,
        "naive_top_doc": naive,
        "advanced_top_doc": advanced_docs[0]["text"],
        "different": naive != advanced_docs[0]["text"]
    })

results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'query': 'how do transformers encode meaning?',
  'naive_top_doc': 'BERT is a bidirectional transformer encoder model.',
  'advanced_top_doc': 'Transformers use self-attention mechanisms to process sequences in parallel.',
  'different': True},
 {'query': 'optimization techniques for training',
  'naive_top_doc': 'Gradient descent is an optimization algorithm used to minimize loss functions.',
  'advanced_top_doc': 'Backpropagation computes gradients for neural network training.',
  'different': True},
 {'query': 'what is attention in deep learning?',
  'naive_top_doc': 'Attention allows models to focus on important tokens in a sequence.',
  'advanced_top_doc': 'Attention allows models to focus on important tokens in a sequence.',
  'different': False}]

This cell contains the final comparison table summarizing results.

It highlights differences between Naïve and Advanced RAG outputs and supports analysis of performance improvements.

| Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Are they different? |
|---|---|---|---|
| how do transformers encode meaning? | BERT is a bidirectional transformer encoder model. | Transformers use self-attention mechanisms to process sequences in parallel. | Yes |
| optimization techniques for training | Gradient descent is an optimization algorithm used to minimize loss functions. | Backpropagation computes gradients for neural network training. | Yes |
| what is attention in deep learning? | Attention allows models to focus on important tokens in a sequence. | Attention allows models to focus on important tokens in a sequence. | No |

This notebook implements an Advanced Retrieval-Augmented Generation (RAG) system that improves upon Naïve RAG by incorporating:

- Hybrid Retrieval (BM25 + SBERT)
- Query Expansion
- Cross-Encoder Re-Ranking

The system is evaluated by comparing it with a baseline Naïve RAG approach.